# LeetCode 1608 — Special Array With X Elements Greater Than or Equal X bằng Python

**Link đề:** https://leetcode.com/problems/special-array-with-x-elements-greater-than-or-equal-x/
**Độ khó:** Easy · **Chủ đề:** Array, Binary Search, Sorting

```python
class Solution:
    def specialArray(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Đọc kỹ đề "có **đúng** `x` số `>= x`" và hiểu vì sao chỉ cần thử `x` trong `0..n`.
2. Đi qua 4 cách: thử mọi `x` `O(n²)` → sắp xếp rồi quét `O(n log n)` → **đếm (counting) + cộng dồn từ phải** `O(n)` → **binary search** trên `x`.
3. Học `sorted(..., reverse=True)`, `bisect_left` và kỹ thuật đếm theo "thùng" với `min(v, n)`.
4. Chứng minh `x` (nếu có) là **duy nhất** bằng tính đơn điệu — chìa khóa để binary search.
5. Trace, test đối chiếu nhiều cách, và luyện biến thể (H-Index).

## 1. Đề bài

Cho mảng `nums` gồm các số nguyên **không âm**. `nums` gọi là **đặc biệt** nếu tồn tại số `x` sao cho có **đúng `x` phần tử** của `nums` **lớn hơn hoặc bằng `x`**.

- `x` **không bắt buộc** phải là một phần tử của `nums`.
- Trả về `x` nếu mảng đặc biệt, ngược lại trả `-1`. (Đề đảm bảo nếu có thì `x` là duy nhất.)

| `nums` | Kết quả | Vì sao |
|---|:---:|---|
| `[3, 5]` | `2` | có 2 số (`3`, `5`) `>= 2` |
| `[0, 0]` | `-1` | x=0: có 2 số ≥ 0 (cần 0); x=1: có 0 (cần 1); x=2: có 0 (cần 2) |
| `[0, 4, 3, 0, 4]` | `3` | có 3 số (`4`, `3`, `4`) `>= 3` |

**Constraints:** `1 <= nums.length <= 100`; `0 <= nums[i] <= 1000`.

In [1]:
examples = [([3, 5], 2), ([0, 0], -1), ([0, 4, 3, 0, 4], 3)]

def count_ge(nums, x):
    # số phần tử >= x
    return sum(1 for v in nums if v >= x)

for nums, expected in examples:
    print(nums, "| count_ge cho x = 0..n:", [count_ge(nums, x) for x in range(len(nums) + 1)], "| kỳ vọng", expected)

[3, 5] | count_ge cho x = 0..n: [2, 2, 2] | kỳ vọng 2
[0, 0] | count_ge cho x = 0..n: [2, 0, 0] | kỳ vọng -1
[0, 4, 3, 0, 4] | count_ge cho x = 0..n: [5, 3, 3, 3, 2, 0] | kỳ vọng 3


Nhìn bảng `count_ge`: ta tìm `x` mà **`count_ge(nums, x) == x`**. Ví dụ 3: `x = 0..5` cho `[5, 3, 3, 3, 2, 0]` → tại `x = 3` có `count = 3` ✓.

## 2. Nền tảng Python

### 2.1. Generator expression để đếm: `sum(1 for v in nums if điều_kiện)`

Sinh số `1` cho mỗi phần tử thỏa điều kiện rồi cộng → số phần tử thỏa. Cũng có thể viết `sum(v >= x for v in nums)` vì `True` cộng như `1`.

In [2]:
nums = [0, 4, 3, 0, 4]
print(sum(1 for v in nums if v >= 3))
print(sum(v >= 3 for v in nums), "| True + True =", True + True)

3
3 | True + True = 2


### 2.2. Vì sao chỉ cần thử `x` từ `0` đến `n`?

Có `n = len(nums)` phần tử, nên `count_ge(nums, x) <= n` với mọi `x`. Nếu `x > n` thì không thể có **đúng `x`** phần tử. Vậy `x ∈ {0, 1, ..., n}`.

Thêm nữa: `x = 0` cần "0 phần tử `>= 0`", nhưng mọi phần tử đều `>= 0` và `n >= 1` → `x = 0` **không bao giờ** đúng. Thực tế chỉ cần `x ∈ {1, ..., n}` (ta vẫn thử cả 0 cho an toàn ở cách brute).

In [3]:
nums = [3, 5]
for x in range(0, len(nums) + 3):
    c = count_ge(nums, x)
    print(f"x={x}: count_ge={c} {'<- khớp!' if c == x else ''}")

x=0: count_ge=2 
x=1: count_ge=2 
x=2: count_ge=2 <- khớp!
x=3: count_ge=2 
x=4: count_ge=1 


### 2.3. `sorted(nums, reverse=True)` — sắp xếp giảm dần

`reverse=True` đảo thứ tự thành lớn → nhỏ. Sau khi sắp giảm dần, "các phần tử `>= x`" luôn nằm ở **đầu** mảng.

In [4]:
nums = [0, 4, 3, 0, 4]
print("tăng dần:", sorted(nums))
print("giảm dần:", sorted(nums, reverse=True))
print("gốc không đổi:", nums)

tăng dần: [0, 0, 3, 4, 4]
giảm dần: [4, 4, 3, 0, 0]
gốc không đổi: [0, 4, 3, 0, 4]


### 2.4. `bisect_left` — tìm vị trí chèn trong mảng đã sắp **tăng dần**

`bisect.bisect_left(a, x)` trả về chỉ số đầu tiên `i` mà `a[i] >= x` (thư viện chuẩn `bisect`, chạy `O(log n)`). Do đó **số phần tử `>= x` = `len(a) - bisect_left(a, x)`**.

In [5]:
from bisect import bisect_left
a = sorted([0, 4, 3, 0, 4])   # [0, 0, 3, 4, 4]
for x in [0, 1, 3, 4, 5]:
    i = bisect_left(a, x)
    print(f"x={x}: bisect_left={i} -> số phần tử >= x = {len(a) - i}")

x=0: bisect_left=0 -> số phần tử >= x = 5
x=1: bisect_left=2 -> số phần tử >= x = 3
x=3: bisect_left=2 -> số phần tử >= x = 3
x=4: bisect_left=3 -> số phần tử >= x = 2
x=5: bisect_left=5 -> số phần tử >= x = 0


## 3. Cách 1 — Brute Force: thử mọi `x` từ 0 tới n

Đúng như gợi ý của LeetCode: với mỗi `x`, đếm số phần tử `>= x`; nếu bằng `x` thì trả về.

Mỗi lần đếm `O(n)`, có `n + 1` giá trị `x` → `O(n²)`. Với `n ≤ 100` hoàn toàn ổn.

In [6]:
def special_brute(nums: list[int]) -> int:
    n = len(nums)
    for x in range(n + 1):
        if count_ge(nums, x) == x:
            return x
    return -1

for nums, expected in examples:
    print(nums, "->", special_brute(nums), "| kỳ vọng", expected)

[3, 5] -> 2 | kỳ vọng 2
[0, 0] -> -1 | kỳ vọng -1
[0, 4, 3, 0, 4] -> 3 | kỳ vọng 3


### 3.1. Vì sao `x` là duy nhất? — tính đơn điệu

Gọi `f(x) = count_ge(nums, x)`.
- Khi `x` tăng, điều kiện `v >= x` khó hơn → `f(x)` **không tăng** (giữ nguyên hoặc giảm).
- Còn `g(x) = x` **tăng thật sự**.

Một hàm không tăng và một hàm tăng ngặt cắt nhau **tối đa một lần** → nếu có `x` thỏa `f(x) == x` thì nó duy nhất.

Hệ quả quan trọng: `h(x) = f(x) - x` **giảm ngặt**. Khi `h(x) > 0` thì đáp án (nếu có) nằm **bên phải**, khi `h(x) < 0` thì nằm **bên trái** → dùng được **binary search**.

In [7]:
nums = [0, 4, 3, 0, 4]
for x in range(len(nums) + 1):
    f = count_ge(nums, x)
    print(f"x={x}: f(x)={f}, f(x)-x={f - x:+d}")

x=0: f(x)=5, f(x)-x=+5
x=1: f(x)=3, f(x)-x=+2
x=2: f(x)=3, f(x)-x=+1
x=3: f(x)=3, f(x)-x=+0
x=4: f(x)=2, f(x)-x=-2
x=5: f(x)=0, f(x)-x=-5


## 4. Cách 2 — Sắp xếp giảm dần rồi quét

Sau khi sắp giảm dần `s = sorted(nums, reverse=True)`, đánh chỉ số từ 0:

- Có **ít nhất** `x` phần tử `>= x` ⟺ `s[x-1] >= x` (phần tử thứ `x` vẫn `>= x`).
- Có **không quá** `x` phần tử `>= x` ⟺ `x == n` hoặc `s[x] < x` (phần tử thứ `x+1` đã `< x`).

Kết hợp: `x` hợp lệ ⟺ `s[x-1] >= x` **và** (`x == n` **hoặc** `s[x] < x`).

In [8]:
s = sorted([0, 4, 3, 0, 4], reverse=True)
print("s =", s)
n = len(s)
for x in range(1, n + 1):
    at_least = s[x - 1] >= x
    at_most = (x == n) or (s[x] < x)
    print(f"x={x}: s[x-1]={s[x-1]} >= {x}? {at_least} | x==n hoặc s[x]<x? {at_most} -> {'HỢP LỆ' if at_least and at_most else '-'}")

s = [4, 4, 3, 0, 0]
x=1: s[x-1]=4 >= 1? True | x==n hoặc s[x]<x? False -> -
x=2: s[x-1]=4 >= 2? True | x==n hoặc s[x]<x? False -> -
x=3: s[x-1]=3 >= 3? True | x==n hoặc s[x]<x? True -> HỢP LỆ
x=4: s[x-1]=0 >= 4? False | x==n hoặc s[x]<x? True -> -
x=5: s[x-1]=0 >= 5? False | x==n hoặc s[x]<x? True -> -


In [9]:
def special_sort(nums: list[int]) -> int:
    s = sorted(nums, reverse=True)
    n = len(s)
    for x in range(1, n + 1):
        if s[x - 1] >= x and (x == n or s[x] < x):
            return x
    return -1

print([special_sort(nums) for nums, _ in examples])   # [2, -1, 3]

[2, -1, 3]


Chú ý **đánh giá ngắn mạch** trong `(x == n or s[x] < x)`: khi `x == n` đúng, Python **không** đọc `s[x]` (vốn sẽ vượt biên). Thứ tự hai vế rất quan trọng.

## 5. Nút thắt và insight cho `O(n)` — đếm theo "thùng"

Sắp xếp tốn `O(n log n)`. Nhưng ta chỉ cần biết `f(x)` cho `x ∈ 1..n`, mà giá trị nào `> n` thì cũng chỉ đóng vai trò "**rất lớn**" (luôn `>= x` với mọi `x <= n`).

**Insight:** gom mọi giá trị `>= n` vào chung thùng `n`:

```python
cnt = [0] * (n + 1)
for v in nums:
    cnt[min(v, n)] += 1
```

Sau đó `f(x) = cnt[x] + cnt[x+1] + ... + cnt[n]` — tổng **hậu tố (suffix sum)**. Duyệt `x` từ `n` xuống `1`, cộng dồn dần là có `f(x)` từng bước trong `O(1)`.

In [10]:
nums = [0, 4, 3, 0, 4, 1000]
n = len(nums)
cnt = [0] * (n + 1)
for v in nums:
    cnt[min(v, n)] += 1
print("n =", n, "| cnt =", cnt, "(1000 rơi vào thùng", n, ")")
suffix = 0
for x in range(n, -1, -1):
    suffix += cnt[x]
    print(f"x={x}: f(x) = {suffix}, bằng count_ge? {suffix == count_ge(nums, x)}")

n = 6 | cnt = [2, 0, 0, 1, 2, 0, 1] (1000 rơi vào thùng 6 )
x=6: f(x) = 1, bằng count_ge? True
x=5: f(x) = 1, bằng count_ge? True
x=4: f(x) = 3, bằng count_ge? True
x=3: f(x) = 4, bằng count_ge? True
x=2: f(x) = 4, bằng count_ge? True
x=1: f(x) = 4, bằng count_ge? True
x=0: f(x) = 6, bằng count_ge? True


## 6. Cách 3 (lời giải chính) `O(n)` — giảng từng đoạn

### 6.1. Đoạn 1: đếm vào thùng
### 6.2. Đoạn 2: duyệt `x` từ `n` xuống `1`, giữ `ge` = số phần tử `>= x`
### 6.3. Đoạn 3: gặp `ge == x` thì trả `x`; hết vòng thì `-1`

In [11]:
def special_counting(nums: list[int]) -> int:
    n = len(nums)
    cnt = [0] * (n + 1)
    for v in nums:                     # Đoạn 1
        cnt[min(v, n)] += 1
    ge = 0
    for x in range(n, 0, -1):          # Đoạn 2: x = n, n-1, ..., 1
        ge += cnt[x]                   # ge = số phần tử >= x
        if ge == x:                    # Đoạn 3
            return x
    return -1

for nums, expected in examples:
    print(nums, "->", special_counting(nums), "| kỳ vọng", expected)

[3, 5] -> 2 | kỳ vọng 2
[0, 0] -> -1 | kỳ vọng -1
[0, 4, 3, 0, 4] -> 3 | kỳ vọng 3


Vì sao duyệt **từ lớn xuống nhỏ**? Vì `f(x) = f(x+1) + cnt[x]`: biết `f` của `x+1` thì thêm thùng `x` là có `f(x)`. Đi từ trên xuống, mỗi bước chỉ cộng một số.

`range(n, 0, -1)` sinh `n, n-1, ..., 1` (dừng **trước** 0).

In [12]:
print(list(range(5, 0, -1)))

[5, 4, 3, 2, 1]


## 7. TRACE từng bước

Ví dụ 3: `nums = [0, 4, 3, 0, 4]`, `n = 5`.

**Đếm thùng:** `0→2 lần`, `3→1`, `4→2` → `cnt = [2, 0, 0, 1, 2, 0]`.

| x | cnt[x] | ge (sau cộng) | ge == x? |
|:-:|:-:|:-:|:-:|
| 5 | 0 | 0 | không |
| 4 | 2 | 2 | không |
| 3 | 1 | 3 | **có → trả 3** |

In [13]:
def trace_special(nums):
    n = len(nums)
    cnt = [0] * (n + 1)
    for v in nums:
        cnt[min(v, n)] += 1
    print(f"nums={nums}, n={n}, cnt={cnt}")
    ge = 0
    for x in range(n, 0, -1):
        ge += cnt[x]
        hit = ge == x
        print(f"  x={x}: cnt[{x}]={cnt[x]}, ge={ge} -> {'KHỚP, trả ' + str(x) if hit else 'chưa'}")
        if hit:
            return x
    print("  không có x nào -> -1")
    return -1

trace_special([0, 4, 3, 0, 4])
print()
trace_special([0, 0])

nums=[0, 4, 3, 0, 4], n=5, cnt=[2, 0, 0, 1, 2, 0]
  x=5: cnt[5]=0, ge=0 -> chưa
  x=4: cnt[4]=2, ge=2 -> chưa
  x=3: cnt[3]=1, ge=3 -> KHỚP, trả 3

nums=[0, 0], n=2, cnt=[2, 0, 0]
  x=2: cnt[2]=0, ge=0 -> chưa
  x=1: cnt[1]=0, ge=0 -> chưa
  không có x nào -> -1


-1

Một ví dụ "nhảy qua" — `ge` vượt `x` mà chưa từng bằng: `[3, 6, 7, 7, 0]`. Tại `x = 4` có `ge = 3 < 4`, tại `x = 3` có `ge = 4 > 3` → bỏ lỡ, không có đáp án.

In [14]:
trace_special([3, 6, 7, 7, 0])

nums=[3, 6, 7, 7, 0], n=5, cnt=[1, 0, 0, 1, 0, 3]
  x=5: cnt[5]=3, ge=3 -> chưa
  x=4: cnt[4]=0, ge=3 -> chưa
  x=3: cnt[3]=1, ge=4 -> chưa
  x=2: cnt[2]=0, ge=4 -> chưa
  x=1: cnt[1]=0, ge=4 -> chưa
  không có x nào -> -1


-1

## 8. Cách 4 — Binary search trên `x`

Từ mục 3.1: `h(x) = f(x) - x` giảm ngặt. Sắp tăng dần `a`, rồi tính `f(x) = n - bisect_left(a, x)` trong `O(log n)`. Binary search `x` trong `[1, n]`:

- `f(mid) == mid` → trả `mid`.
- `f(mid) > mid` → còn "dư" phần tử → tăng `x`: `lo = mid + 1`.
- `f(mid) < mid` → thiếu → giảm `x`: `hi = mid - 1`.

In [15]:
def special_binary(nums: list[int]) -> int:
    a = sorted(nums)
    n = len(a)
    lo, hi = 1, n
    while lo <= hi:
        mid = (lo + hi) // 2
        f = n - bisect_left(a, mid)
        if f == mid:
            return mid
        if f > mid:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

print([special_binary(nums) for nums, _ in examples])

[2, -1, 3]


In [16]:
def trace_binary(nums):
    a = sorted(nums)
    n = len(a)
    lo, hi = 1, n
    print("a =", a)
    while lo <= hi:
        mid = (lo + hi) // 2
        f = n - bisect_left(a, mid)
        print(f"  lo={lo}, hi={hi}, mid={mid}, f(mid)={f}", "-> trả" if f == mid else ("-> sang phải" if f > mid else "-> sang trái"))
        if f == mid:
            return mid
        lo, hi = (mid + 1, hi) if f > mid else (lo, mid - 1)
    return -1

print("Kết quả:", trace_binary([0, 4, 3, 0, 4]))

a = [0, 0, 3, 4, 4]
  lo=1, hi=5, mid=3, f(mid)=3 -> trả
Kết quả: 3


## 9. Code nộp LeetCode

### 9.1. Cách chính — đếm thùng `O(n)`

In [17]:
class Solution:
    def specialArray(self, nums: list[int]) -> int:
        n = len(nums)
        cnt = [0] * (n + 1)
        for v in nums:
            cnt[min(v, n)] += 1
        ge = 0
        for x in range(n, 0, -1):
            ge += cnt[x]
            if ge == x:
                return x
        return -1

print([Solution().specialArray(nums) for nums, _ in examples])

[2, -1, 3]


### 9.2. Brute Force, sắp xếp, binary search

In [18]:
class SolutionBrute:
    def specialArray(self, nums: list[int]) -> int:
        for x in range(len(nums) + 1):
            if sum(v >= x for v in nums) == x:
                return x
        return -1

class SolutionSort:
    def specialArray(self, nums: list[int]) -> int:
        s = sorted(nums, reverse=True)
        n = len(s)
        for x in range(1, n + 1):
            if s[x - 1] >= x and (x == n or s[x] < x):
                return x
        return -1

class SolutionBinary:
    def specialArray(self, nums: list[int]) -> int:
        a = sorted(nums)
        n = len(a)
        lo, hi = 1, n
        while lo <= hi:
            mid = (lo + hi) // 2
            f = n - bisect_left(a, mid)
            if f == mid:
                return mid
            if f > mid:
                lo = mid + 1
            else:
                hi = mid - 1
        return -1

for cls in [SolutionBrute, SolutionSort, SolutionBinary]:
    print(cls.__name__, [cls().specialArray(nums) for nums, _ in examples])

SolutionBrute [2, -1, 3]
SolutionSort [2, -1, 3]
SolutionBinary [2, -1, 3]


## 10. Vì sao đúng? Độ phức tạp và so sánh

**Cách đếm thùng:** với `1 <= x <= n`, phần tử `v` thỏa `v >= x` ⟺ `min(v, n) >= x` (vì nếu `v >= n` thì `min = n >= x`; nếu `v < n` thì `min = v`). Vậy `f(x) = Σ_{k>=x} cnt[k]`. **Invariant** của vòng lặp: ngay sau `ge += cnt[x]`, `ge == f(x)`. Ta kiểm tra đủ mọi `x ∈ [1, n]` (còn `x = 0` không bao giờ đúng, mục 2.2) → không bỏ sót; trả về `x` đầu tiên gặp, mà `x` là duy nhất (mục 3.1) → không trả nhầm.

**Cách binary search:** đúng nhờ `f(x) - x` giảm ngặt — nửa bị loại chắc chắn không chứa đáp án.

Gọi `n = len(nums)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute Force | `O(n²)` | `O(1)` |
| 2. Sắp xếp + quét | `O(n log n)` | `O(n)` (bản sắp xếp) |
| 3. Đếm thùng | **`O(n)`** | `O(n)` |
| 4. Sắp xếp + binary search | `O(n log n)` (sắp xếp chiếm ưu thế; phần tìm `O(log² n)`) | `O(n)` |

## 11. Bộ test

1. Ví dụ đề.
2. Biên: `[0]` → -1, `[1]` → 1, `[1000]` → 1, `[2, 2]` → 2, `[1, 1]` → -1.
3. Mọi phần tử bằng nhau và lớn: `[1000]*100` → 100.
4. "Nhảy qua": `[3, 6, 7, 7, 0]` → -1.
5. Đối chiếu 8 hàm trên mảng ngẫu nhiên; kiểm tra định nghĩa trực tiếp.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS ({actual})")

solvers = [special_brute, special_sort, special_counting, special_binary,
           Solution().specialArray, SolutionBrute().specialArray, SolutionSort().specialArray, SolutionBinary().specialArray]
cases = [
    ("ví dụ 1", [3, 5], 2),
    ("ví dụ 2", [0, 0], -1),
    ("ví dụ 3", [0, 4, 3, 0, 4], 3),
    ("một số 0", [0], -1),
    ("một số 1", [1], 1),
    ("một số 1000", [1000], 1),
    ("[2, 2]", [2, 2], 2),
    ("[1, 1]", [1, 1], -1),
    ("100 số 1000", [1000] * 100, 100),
    ("nhảy qua", [3, 6, 7, 7, 0], -1),
    ("x không thuộc mảng", [3, 3, 3], 3),
    ("x không thuộc mảng 2", [5, 5, 0, 0], 2),
]
for name, nums, expected in cases:
    for f in solvers:
        assert f(nums) == expected, (name, f, f(nums))
    check(name, special_counting(nums), expected)

ví dụ 1                    PASS (2)
ví dụ 2                    PASS (-1)
ví dụ 3                    PASS (3)
một số 0                   PASS (-1)
một số 1                   PASS (1)
một số 1000                PASS (1)
[2, 2]                     PASS (2)
[1, 1]                     PASS (-1)
100 số 1000                PASS (100)
nhảy qua                   PASS (-1)
x không thuộc mảng         PASS (3)
x không thuộc mảng 2       PASS (2)


In [20]:
import random
rng = random.Random(1608)
found = 0
for _ in range(1000):
    n = rng.randint(1, 12)
    nums = [rng.randint(0, rng.choice([3, 10, 1000])) for _ in range(n)]
    ans = special_brute(nums)
    for f in solvers:
        assert f(nums) == ans
    if ans != -1:
        found += 1
        assert count_ge(nums, ans) == ans
    # tính duy nhất: không có x nào khác thỏa
    assert sum(1 for x in range(n + 1) if count_ge(nums, x) == x) <= 1
print(f"1000 mảng ngẫu nhiên: 8 hàm khớp; {found} mảng đặc biệt; x luôn duy nhất")

1000 mảng ngẫu nhiên: 8 hàm khớp; 691 mảng đặc biệt; x luôn duy nhất


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ thử `x` là phần tử của mảng

Đề nói rõ `x` **không cần** thuộc `nums`. Ví dụ `[3, 5]` → `x = 2` không có trong mảng.

In [21]:
def wrong_only_elements(nums):
    for x in set(nums):
        if count_ge(nums, x) == x:
            return x
    return -1

print("sai:", wrong_only_elements([3, 5]), "| đúng:", special_counting([3, 5]))

sai: -1 | đúng: 2


### Lỗi 2: Đếm `>` thay vì `>=`

In [22]:
def wrong_strict(nums):
    for x in range(len(nums) + 1):
        if sum(v > x for v in nums) == x:
            return x
    return -1

print("sai:", wrong_strict([0, 4, 3, 0, 4]), "| đúng:", special_counting([0, 4, 3, 0, 4]))

sai: -1 | đúng: 3


### Lỗi 3: Quên gom giá trị lớn — `cnt[v] += 1` với `cnt` dài `n + 1`

Giá trị có thể tới `1000` trong khi `n` nhỏ → `IndexError`. Phải dùng `min(v, n)`.

In [23]:
def wrong_no_min(nums):
    n = len(nums)
    cnt = [0] * (n + 1)
    for v in nums:
        cnt[v] += 1
    return cnt

try:
    wrong_no_min([3, 5])
except IndexError as e:
    print("IndexError:", e, "-> dùng cnt[min(v, n)]")

IndexError: list index out of range -> dùng cnt[min(v, n)]


### Lỗi 4: Ở cách sắp xếp, đọc `s[x]` khi `x == n`

Viết `s[x] < x and ...` (đảo thứ tự) → `IndexError` khi `x == n`. Đặt `x == n` **trước** trong `or`.

In [24]:
def wrong_order(nums):
    s = sorted(nums, reverse=True)
    n = len(s)
    for x in range(1, n + 1):
        if s[x - 1] >= x and (s[x] < x or x == n):
            return x
    return -1

try:
    wrong_order([3, 5])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


## 13. Mẹo nhận dạng pattern

- "Có **đúng / ít nhất** `x` phần tử `>= x`" → nghĩ tới **H-Index** (LeetCode 274/275): sắp xếp giảm dần hoặc đếm thùng với `min(v, n)`.
- Một hàm đếm **đơn điệu** theo ngưỡng (`count_ge` giảm khi ngưỡng tăng) → có thể **binary search trên đáp án**.
- Giá trị lớn nhưng chỉ cần so với `n` → **cắt về `n`** để đếm trong `O(n)`.
- Bài liên quan: 274 H-Index, 275 H-Index II, 1011/875 (binary search trên đáp án), 2389 Longest Subsequence With Limited Sum.

## 14. Biến thể

### 14.1. H-Index: `x` lớn nhất sao cho có **ít nhất** `x` phần tử `>= x`

Khác bài gốc ở chữ "ít nhất" → luôn có đáp án (ít nhất là 0).

In [25]:
def h_index(citations: list[int]) -> int:
    n = len(citations)
    cnt = [0] * (n + 1)
    for v in citations:
        cnt[min(v, n)] += 1
    ge = 0
    for x in range(n, -1, -1):
        ge += cnt[x]
        if ge >= x:
            return x
    return 0

print(h_index([3, 0, 6, 1, 5]), h_index([1, 3, 1]), h_index([0, 0]))   # 3 1 0

3 1 0


### 14.2. Đếm số `x` có **ít nhất** `x` phần tử `>= x`

Vì `f(x) - x` giảm ngặt, các `x` thỏa `f(x) >= x` tạo thành một đoạn liên tiếp `0..H` → đáp án là `H + 1`.

In [26]:
def count_at_least(nums):
    return sum(1 for x in range(len(nums) + 1) if count_ge(nums, x) >= x)

nums = [3, 0, 6, 1, 5]
print(count_at_least(nums), "=", h_index(nums) + 1)

4 = 4


### 14.3. Ngưỡng "nhỏ hơn hoặc bằng": có đúng `x` phần tử `<= x`?

Đổi chiều đơn điệu: `f(x) = count_le(x)` **tăng** theo `x`, còn `x` cũng tăng → **không** còn đảm bảo duy nhất! Thử bằng code:

In [27]:
def all_x_le(nums):
    return [x for x in range(len(nums) + 1) if sum(v <= x for v in nums) == x]

print(all_x_le([1, 2, 3]))    # nhiều đáp án
print(all_x_le([0, 0, 5]))

[0, 1, 2, 3]
[2]


## 15. Bài tập tự luyện (TODO)

**Bài 1 — tính tay.** Với `nums = [5, 5, 0, 0]`, lập bảng `x`, `count_ge` cho `x = 0..4` và chỉ ra đáp án.

**Bài 2 — đoán output.** `bisect_left([1, 3, 3, 7], 3)` và `bisect_left([1, 3, 3, 7], 4)` bằng bao nhiêu?

### Bài 3 — Điền TODO: cách brute force

In [28]:
def exercise_brute(nums: list[int]) -> int:
    # TODO 1: duyệt x từ 0 tới len(nums)
    # TODO 2: đếm số phần tử >= x; nếu bằng x thì return x
    # TODO 3: return -1
    pass

# assert exercise_brute([3, 5]) == 2

### Bài 4 — Tự viết cách đếm thùng `O(n)`

In [29]:
def exercise_counting(nums: list[int]) -> int:
    # TODO: cnt[min(v, n)] += 1; duyệt x từ n xuống 1
    pass

# assert exercise_counting([0, 4, 3, 0, 4]) == 3
# assert exercise_counting([0, 0]) == -1

### Bài 5 — Biến thể: H-Index bằng sắp xếp giảm dần

In [30]:
def exercise_h_index_sort(citations: list[int]) -> int:
    # TODO: sắp giảm dần; h tăng khi s[h] >= h + 1
    pass

# assert exercise_h_index_sort([3, 0, 6, 1, 5]) == 3

### Bài 6 — Mini challenge: binary search không dùng `bisect`

Viết lại `special_binary` nhưng tự viết hàm đếm số phần tử `>= x` trên mảng đã sắp tăng bằng binary search.

In [31]:
def exercise_binary_manual(nums: list[int]) -> int:
    # TODO
    pass

# assert exercise_binary_manual([0, 4, 3, 0, 4]) == 3

## 16. Gợi ý 3 mức

**Bài 1:** M1: đếm số ≥ 0, ≥ 1, ... M2: `[4, 2, 2, 2, 2]`. M3: tìm vị trí `count == x`.

**Bài 2:** M1: chỉ số đầu tiên có `a[i] >= x`. M2: phần tử 3 đầu tiên ở chỉ số 1. M3: 7 là phần tử đầu tiên ≥ 4.

**Bài 3:** M1: làm đúng định nghĩa. M2: `sum(v >= x for v in nums)`. M3: vòng `for x in range(len(nums) + 1)`.

**Bài 4:** M1: gom giá trị lớn về `n`. M2: `ge += cnt[x]` trong vòng giảm dần. M3: `if ge == x: return x`.

**Bài 5:** M1: sau khi sắp giảm, `s[i]` là bài thứ `i+1` nhiều trích dẫn nhất. M2: `while h < n and s[h] >= h + 1: h += 1`. M3: trả `h`.

**Bài 6:** M1: tìm chỉ số đầu tiên `a[i] >= x`. M2: `lo, hi = 0, n`; `while lo < hi`. M3: `if a[mid] < x: lo = mid + 1 else: hi = mid`.

## 17. Đáp án tham khảo

**Bài 1:** `count_ge` cho `x = 0..4` là `[4, 2, 2, 2, 2]` → `x = 2` (`count_ge = 2`). Đáp án `2`.

**Bài 2:** `1` và `3`.

In [32]:
def answer_brute(nums):
    for x in range(len(nums) + 1):
        if sum(v >= x for v in nums) == x:
            return x
    return -1

def answer_counting(nums):
    n = len(nums)
    cnt = [0] * (n + 1)
    for v in nums:
        cnt[min(v, n)] += 1
    ge = 0
    for x in range(n, 0, -1):
        ge += cnt[x]
        if ge == x:
            return x
    return -1

def answer_h_index_sort(citations):
    s = sorted(citations, reverse=True)
    h = 0
    while h < len(s) and s[h] >= h + 1:
        h += 1
    return h

def first_ge(a, x):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

def answer_binary_manual(nums):
    a = sorted(nums)
    n = len(a)
    lo, hi = 1, n
    while lo <= hi:
        mid = (lo + hi) // 2
        f = n - first_ge(a, mid)
        if f == mid:
            return mid
        if f > mid:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

assert [count_ge([5, 5, 0, 0], x) for x in range(5)] == [4, 2, 2, 2, 2]
assert answer_brute([5, 5, 0, 0]) == 2
assert bisect_left([1, 3, 3, 7], 3) == 1 and bisect_left([1, 3, 3, 7], 4) == 3
assert answer_brute([3, 5]) == 2
assert answer_counting([0, 4, 3, 0, 4]) == 3 and answer_counting([0, 0]) == -1
assert answer_h_index_sort([3, 0, 6, 1, 5]) == 3
assert answer_binary_manual([0, 4, 3, 0, 4]) == 3
for _ in range(500):
    nums = [rng.randint(0, 15) for _ in range(rng.randint(1, 12))]
    assert answer_brute(nums) == answer_counting(nums) == answer_binary_manual(nums) == special_brute(nums)
    assert answer_h_index_sort(nums) == h_index(nums)
    a = sorted(nums)
    for x in range(0, 17):
        assert first_ge(a, x) == bisect_left(a, x)
print("Đáp án tham khảo: PASS toàn bộ!")

Đáp án tham khảo: PASS toàn bộ!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def specialArray(self, nums: list[int]) -> int:
        n = len(nums)
        cnt = [0] * (n + 1)
        for v in nums:
            cnt[min(v, n)] += 1        # giá trị >= n gom về thùng n
        ge = 0
        for x in range(n, 0, -1):      # đi từ lớn xuống nhỏ
            ge += cnt[x]               # ge = số phần tử >= x
            if ge == x:
                return x
        return -1
```

**Câu thần chú:** *"x chỉ trong 1..n; đếm thùng, cộng dồn từ phải; f giảm, x tăng → gặp nhau tối đa 1 lần."*

**Checklist trước khi submit:**
- [ ] Thử cả `x` **không** thuộc mảng (duyệt `1..n`, không duyệt phần tử)?
- [ ] Dùng `>=` chứ không phải `>`?
- [ ] Gom `min(v, n)` để không tràn chỉ số?
- [ ] Cách sắp xếp: kiểm tra `x == n` trước khi đọc `s[x]`?
- [ ] Đã thử `[0, 0]` → `-1` và `[3, 5]` → `2`?